In [0]:
dbutils.widgets.text("fecha_corte", "2026-11-05")
dbutils.widgets.dropdown("modo", "incremental", ["incremental", "full", "reproceso"])
dbutils.widgets.dropdown("env", "dev", ["dev", "qa", "prod"])
dbutils.widgets.text("run_id", "")   # el job lo envía; vacío = se genera uno nuevo

In [0]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.common.config import PipelineConfig
from src.common.control import ejecucion
from src.bronze.ingesta import ingerir_todo

params = dict(
    fecha_corte=dbutils.widgets.get("fecha_corte"),
    modo=dbutils.widgets.get("modo"),
    env=dbutils.widgets.get("env"),
)
if dbutils.widgets.get("run_id"):
    params["run_id"] = dbutils.widgets.get("run_id")

cfg = PipelineConfig(**params)

with ejecucion(spark, cfg, "bronze"):
    resultado = ingerir_todo(spark, cfg)

display(spark.createDataFrame(resultado))

In [0]:
%sql
SELECT 'clientes' AS tabla, _fecha_corte, _run_id, COUNT(*) AS filas FROM workspace.default.bronze_clientes GROUP BY ALL
UNION ALL SELECT 'polizas', _fecha_corte, _run_id, COUNT(*) FROM workspace.default.bronze_polizas GROUP BY ALL
UNION ALL SELECT 'recaudos', _fecha_corte, _run_id, COUNT(*) FROM workspace.default.bronze_recaudos GROUP BY ALL
UNION ALL SELECT 'siniestros', _fecha_corte, _run_id, COUNT(*) FROM workspace.default.bronze_siniestros GROUP BY ALL
UNION ALL SELECT 'movimientos', _fecha_corte, _run_id, COUNT(*) FROM workspace.default.bronze_movimientos GROUP BY ALL

In [0]:
%sql
-- ¿Cuál fue la última ejecución exitosa?
SELECT run_id, etapa, fecha_corte, estado, inicio_ts, round(duracion_seg,1) AS seg, error_mensaje
FROM workspace.default.ctl_ejecuciones
ORDER BY inicio_ts DESC;

In [0]:
%sql
-- ¿Qué se cargó y cuánto sumaba? (cuadre)
SELECT run_id, tabla, filas_leidas, filas_escritas, filas_rechazadas, totales_monetarios
FROM workspace.default.ctl_cargas
ORDER BY registrado_ts DESC;